# BEAD Random Forest — Drop `jobs_per_location`, Add CAI

**Reviewer comment #23**: `jobs_per_location` is not hypothesis-motivated, not listed in Table 1,
and likely acts as a proxy for technology type (fiber builds generate more jobs per location
than satellite/fixed-wireless). Drop it.

**Replacement**: incorporate CAI (Community Anchor Institution) count from the `loc_cai` table.
CAI locations may have different infrastructure requirements and costs, making this a
hypothesis-motivated predictor of funding per location.

Note: some `location_id` values in `loc_cai` are empty — this is expected and not a data error.

**Three models compared**:
1. **6-feat (original)** — includes `jobs_per_location` (the final model from bead_rf_final.ipynb)
2. **5-feat (no jobs)** — drop `jobs_per_location`, no replacement
3. **6-feat (CAI swap)** — drop `jobs_per_location`, add `pct_cai` from `loc_cai`

In [ ]:
from google.cloud import bigquery
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

client = bigquery.Client(project='broadband-data')
print('Connected to BigQuery: broadband-data')

In [ ]:
# Load all data sources
df_projects = client.query("""
SELECT project_id, state, bead_support, estimated_miles_aerial_fiber,
       estimated_miles_buried_fiber, estimated_jobs, project_type, priority_broadband_project
FROM `broadband-data.fp_approved.deployment_projects`
""").to_dataframe()

df_locations = client.query("""
SELECT project_id, COUNT(*) AS funded_locations,
       SAFE_CAST(APPROX_TOP_COUNT(CAST(technology AS STRING), 1)[OFFSET(0)].value AS FLOAT64) AS technology,
       AVG(CAST(low_latency AS INT64)) AS avg_latency
FROM `broadband-data.fp_approved.locations`
GROUP BY project_id
""").to_dataframe()

# CAI locations per project.
# Some location_ids in loc_cai are empty — expected, not a bug.
# COUNT(*) captures all CAI rows regardless of whether location_id is populated.
df_cai = client.query("""
SELECT project_id, COUNT(*) AS cai_count
FROM `broadband-data.fp_approved.loc_cai`
GROUP BY project_id
""").to_dataframe()

state_name_to_abbr = {
    'Alabama': 'AL', 'Alaska': 'AK', 'Arizona': 'AZ', 'Arkansas': 'AR', 'California': 'CA',
    'Colorado': 'CO', 'Connecticut': 'CT', 'Delaware': 'DE', 'Florida': 'FL', 'Georgia': 'GA',
    'Hawaii': 'HI', 'Idaho': 'ID', 'Illinois': 'IL', 'Indiana': 'IN', 'Iowa': 'IA',
    'Kansas': 'KS', 'Kentucky': 'KY', 'Louisiana': 'LA', 'Maine': 'ME', 'Maryland': 'MD',
    'Massachusetts': 'MA', 'Michigan': 'MI', 'Minnesota': 'MN', 'Mississippi': 'MS',
    'Missouri': 'MO', 'Montana': 'MT', 'Nebraska': 'NE', 'Nevada': 'NV', 'New_Hampshire': 'NH',
    'New_Jersey': 'NJ', 'New_Mexico': 'NM', 'New_York': 'NY', 'North_Carolina': 'NC',
    'North_Dakota': 'ND', 'Ohio': 'OH', 'Oklahoma': 'OK', 'Oregon': 'OR', 'Pennsylvania': 'PA',
    'Rhode_Island': 'RI', 'South_Carolina': 'SC', 'South_Dakota': 'SD', 'Tennessee': 'TN',
    'Texas': 'TX', 'Utah': 'UT', 'Vermont': 'VT', 'Virginia': 'VA', 'Washington': 'WA',
    'West_Virginia': 'WV', 'Wisconsin': 'WI', 'Wyoming': 'WY', 'District_of_Columbia': 'DC'
}
df_nbm = client.query("""
SELECT state, COUNT(DISTINCT frn) AS state_num_providers
FROM `broadband-data.fcc_bdc.nbm_hive` GROUP BY state
""").to_dataframe()
df_nbm['state'] = df_nbm['state'].map(state_name_to_abbr)
df_nbm = df_nbm.dropna(subset=['state'])

df_state_pop = client.query("""
SELECT stateabbr AS state, SUM(pop2020) AS state_population
FROM `broadband-data.fcc_block_level_pop.us2020` GROUP BY stateabbr
""").to_dataframe()

print(f'Projects: {df_projects.shape}')
print(f'Locations: {df_locations.shape}')
print(f'CAI rows: {df_cai.shape[0]} projects with CAI locations')

In [ ]:
# Merge + feature engineering
df = df_projects.merge(df_locations, on='project_id', how='left')
df = df.merge(df_nbm, on='state', how='left')
df = df.merge(df_state_pop, on='state', how='left')
df = df.merge(df_cai, on='project_id', how='left')  # left join: projects with no CAI get NaN -> 0

df['funded_locations'] = df['funded_locations'].fillna(0)
df['technology'] = df['technology'].fillna(0)
df['state_num_providers'] = df['state_num_providers'].fillna(0)
df['state_population'] = df['state_population'].fillna(0)
df['cai_count'] = df['cai_count'].fillna(0)  # 0 CAI locations if project not in loc_cai

df['total_fiber_miles'] = df['estimated_miles_aerial_fiber'].fillna(0) + df['estimated_miles_buried_fiber'].fillna(0)
df['miles_per_location'] = df['total_fiber_miles'] / df['funded_locations'].replace(0, np.nan)
df['miles_per_location'] = df['miles_per_location'].fillna(0)

df['jobs_per_location'] = df['estimated_jobs'] / df['funded_locations'].replace(0, np.nan)
df['jobs_per_location'] = df['jobs_per_location'].fillna(0)

# Fraction of funded locations that are CAI
df['pct_cai'] = df['cai_count'] / df['funded_locations'].replace(0, np.nan)
df['pct_cai'] = df['pct_cai'].fillna(0)

df['funding_per_location'] = df['bead_support'] / df['funded_locations'].replace(0, np.nan)
df = df.dropna(subset=['funding_per_location'])
low = df['funding_per_location'].quantile(0.025)
high = df['funding_per_location'].quantile(0.975)
df = df[(df['funding_per_location'] >= low) & (df['funding_per_location'] <= high)]

df['log_funding'] = np.log1p(df['funding_per_location'])

print(f'Samples after filtering: {df.shape[0]}')
print(f'Projects with any CAI:   {(df["cai_count"] > 0).sum()}')
print(f'pct_cai stats:\n{df["pct_cai"].describe().round(4)}')

In [ ]:
# ── Reviewer #23: does jobs_per_location proxy for technology type? ──────────
tech_labels = {0: "Unspecified", 40: "Cable", 50: "Fiber", 61: "Satellite", 71: "Fixed Wireless"}
df["tech_name"] = df["technology"].map(tech_labels).fillna("Other")
df["is_fiber"] = (df["technology"] == 50).astype(int)

# Mean jobs_per_location by technology
tech_jobs = df.groupby("tech_name")["jobs_per_location"].agg(["mean", "median", "count"]).round(2)
tech_jobs.columns = ["Mean jobs/loc", "Median jobs/loc", "N projects"]
tech_jobs = tech_jobs.sort_values("Mean jobs/loc", ascending=False)
print("jobs_per_location by technology:")
print(tech_jobs.to_string())

# Point-biserial correlation: jobs_per_location vs is_fiber
from scipy import stats
r, p = stats.pointbiserialr(df["is_fiber"], df["jobs_per_location"])
print(f"
Point-biserial r (is_fiber vs jobs_per_location): {r:.3f}  p={p:.3e}")

# Spearman rank correlation (technology code vs jobs_per_location)
rs, ps = stats.spearmanr(df["technology"], df["jobs_per_location"])
print(f"Spearman r  (technology vs jobs_per_location):   {rs:.3f}  p={ps:.3e}")

# Boxplot
order = tech_jobs.index.tolist()
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

data_by_tech = [df.loc[df["tech_name"] == t, "jobs_per_location"].dropna() for t in order]
axes[0].boxplot(data_by_tech, labels=order, vert=True)
axes[0].set_ylabel("jobs_per_location")
axes[0].set_title("jobs_per_location by Technology Type")
axes[0].tick_params(axis="x", rotation=15)

# Scatter: jobs_per_location vs miles_per_location, colored by tech
colors = {"Fiber": "steelblue", "Satellite": "coral", "Fixed Wireless": "seagreen",
          "Cable": "purple", "Unspecified": "grey", "Other": "grey"}
for tname, grp in df.groupby("tech_name"):
    axes[1].scatter(grp["miles_per_location"], grp["jobs_per_location"],
                    alpha=0.4, s=20, label=tname, color=colors.get(tname, "grey"))
axes[1].set_xlabel("miles_per_location")
axes[1].set_ylabel("jobs_per_location")
axes[1].set_title("jobs/loc vs miles/loc by Technology")
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig("fig_jobs_tech_correlation.pdf", bbox_inches="tight")
plt.savefig("fig_jobs_tech_correlation.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved fig_jobs_tech_correlation.{pdf,png}")

## Train all three models
Same hyperparameters and split throughout for fair comparison.

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
y  = df['log_funding']

def fit_and_eval(feature_cols, label):
    X = df[feature_cols].fillna(0)
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42)
    rf = RandomForestRegressor(n_estimators=200, min_samples_split=5, random_state=42, n_jobs=-1)
    rf.fit(X_tr, y_tr)
    y_pred = rf.predict(X_te)
    r2   = r2_score(y_te, y_pred)
    cv   = cross_val_score(rf, X, y, cv=kf, scoring='r2')
    rmse = np.sqrt(mean_squared_error(np.expm1(y_te), np.expm1(y_pred)))
    mae  = mean_absolute_error(np.expm1(y_te), np.expm1(y_pred))
    print(f'=== {label} ===')
    print(f'  Test R²  (log): {r2:.4f}')
    print(f'  CV R²   (log): {cv.mean():.4f} ± {cv.std():.4f}')
    print(f'  RMSE  (real $): ${rmse:,.0f}')
    print(f'  MAE   (real $): ${mae:,.0f}\n')
    return rf, r2, cv, rmse, mae, X_te, y_te, y_pred

# 1. Original 6-feature final model
feats_6_orig = ['miles_per_location', 'technology', 'jobs_per_location',
                'state_population', 'total_fiber_miles', 'state_num_providers']
rf6, r2_6, cv6, rmse_6, mae_6, Xte6, yte6, yp6 = fit_and_eval(feats_6_orig, '6-feat original (with jobs_per_location)')

# 2. 5-feature model (jobs dropped, no replacement)
feats_5 = ['miles_per_location', 'technology',
           'state_population', 'total_fiber_miles', 'state_num_providers']
rf5, r2_5, cv5, rmse_5, mae_5, Xte5, yte5, yp5 = fit_and_eval(feats_5, '5-feat (jobs_per_location dropped)')

# 3. 6-feature model swapping jobs -> pct_cai
feats_6_cai = ['miles_per_location', 'technology', 'pct_cai',
               'state_population', 'total_fiber_miles', 'state_num_providers']
rf6c, r2_6c, cv6c, rmse_6c, mae_6c, Xte6c, yte6c, yp6c = fit_and_eval(feats_6_cai, '6-feat CAI swap (pct_cai replaces jobs_per_location)')

In [ ]:
# ── Summary comparison table ───────────────────────────────────────────────────
results = pd.DataFrame({
    '6-feat original\n(+jobs_per_location)': [
        f'{r2_6:.4f}',
        f'{cv6.mean():.4f} ± {cv6.std():.4f}',
        f'${rmse_6:,.0f}',
        f'${mae_6:,.0f}',
    ],
    '5-feat\n(jobs dropped)': [
        f'{r2_5:.4f}',
        f'{cv5.mean():.4f} ± {cv5.std():.4f}',
        f'${rmse_5:,.0f}',
        f'${mae_5:,.0f}',
    ],
    '6-feat CAI swap\n(pct_cai replaces jobs)': [
        f'{r2_6c:.4f}',
        f'{cv6c.mean():.4f} ± {cv6c.std():.4f}',
        f'${rmse_6c:,.0f}',
        f'${mae_6c:,.0f}',
    ],
    'Δ (CAI swap vs original)': [
        f'{r2_6c - r2_6:+.4f}',
        f'{cv6c.mean() - cv6.mean():+.4f}',
        f'${rmse_6c - rmse_6:+,.0f}',
        f'${mae_6c - mae_6:+,.0f}',
    ],
}, index=['Test R² (log)', 'CV R² (log)', 'RMSE (real $)', 'MAE (real $)'])

print(results.to_string())

In [ ]:
# ── 3-panel pred vs actual + feature importances for CAI-swap model ────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pred vs actual (CAI-swap model)
axes[0].scatter(yte6c, yp6c, alpha=0.4, s=20, color='steelblue')
lims = [min(yte6c.min(), yp6c.min()), max(yte6c.max(), yp6c.max())]
axes[0].plot(lims, lims, 'r--')
axes[0].set_xlabel('Actual log(Funding/Location)')
axes[0].set_ylabel('Predicted')
axes[0].set_title(f'6-feat CAI swap | R²={r2_6c:.3f}')

# Feature importance
imp_cai = pd.Series(rf6c.feature_importances_, index=feats_6_cai).sort_values()
imp_cai.plot.barh(ax=axes[1], color='steelblue')
axes[1].set_xlabel('Importance')
axes[1].set_title('Feature Importances — 6-feat CAI swap')

plt.tight_layout()
plt.savefig('fig_cai_swap_model.pdf', bbox_inches='tight')
plt.savefig('fig_cai_swap_model.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved fig_cai_swap_model.{pdf,png}')

In [ ]:
# ── Feature importance comparison across all 3 models ─────────────────────────
# Align features for side-by-side display (union of all feature sets)
all_feats = list(dict.fromkeys(feats_6_orig + feats_5 + feats_6_cai))  # preserve order, deduplicate

def imp_series(rf, cols):
    s = pd.Series(dict(zip(cols, rf.feature_importances_)))
    return s.reindex(all_feats, fill_value=0)

imp_df = pd.DataFrame({
    '6-feat original': imp_series(rf6, feats_6_orig),
    '5-feat (no jobs)': imp_series(rf5, feats_5),
    '6-feat CAI swap': imp_series(rf6c, feats_6_cai),
}).sort_values('6-feat original')

fig, ax = plt.subplots(figsize=(11, 5))
x = np.arange(len(imp_df))
w = 0.28
ax.barh(x - w, imp_df['6-feat original'],  w, label='6-feat original', color='coral')
ax.barh(x,     imp_df['5-feat (no jobs)'], w, label='5-feat (no jobs)', color='steelblue')
ax.barh(x + w, imp_df['6-feat CAI swap'],  w, label='6-feat CAI swap',  color='seagreen')
ax.set_yticks(x)
ax.set_yticklabels(imp_df.index)
ax.set_xlabel('RF Gini Importance')
ax.set_title('Feature Importance Across Models')
ax.legend()
plt.tight_layout()
plt.savefig('fig_importance_comparison.pdf', bbox_inches='tight')
plt.savefig('fig_importance_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved fig_importance_comparison.{pdf,png}')

## Interpretation

**`jobs_per_location`**: was the 3rd most important feature in the 6-feature model but is methodologically
unmotivated — bidders are not required to report jobs by a consistent methodology, so the variable
captures technology type (fiber builds report more jobs/location) rather than an independent predictor.
The `technology` feature already encodes this directly.

**`pct_cai`**: Community Anchor Institutions (schools, libraries, hospitals) have specific connectivity
requirements and may require more costly infrastructure. Projects with a higher fraction of CAI
locations have a hypothesis-motivated reason to cost more per location. The variable comes directly
from the FP-approved `loc_cai` table.

**Decision rule**: if the CAI-swap model (6-feat with `pct_cai`) matches or improves on the original
6-feat model, it should be preferred — it replaces a noisy proxy with a substantively motivated feature.